<a href="https://colab.research.google.com/github/kondreddygarivani-bit/project-8/blob/main/Copy_of_W8sample_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install pypdf
!pip install sentence-transformers
!pip install faiss-cpu
!pip install transformers
!pip install torch

In [ ]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from transformers import pipeline


In [ ]:
pdf = PdfReader("stories (1).pdf")

text = ""

for page in pdf.pages:
    text += page.extract_text()

print(text[:1000])

In [ ]:
chunk_size = 1000

chunks = []

for i in range(0, len(text), chunk_size):
    chunks.append(text[i:i+chunk_size])

print("Total Chunks:", len(chunks))

In [ ]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

embeddings = model.encode(chunks)

print(embeddings.shape)

In [ ]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("Vectors Added")

In [ ]:
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))
print(len(chunks))

In [ ]:
embeddings = np.array(embeddings).astype("float32")
query_embedding = np.array(
    embedding_model.encode([question])
).astype("float32")

query_embedding = np.array(
    embedding_model.encode([question])
).astype("float32")

distances, indices = index.search(
    query_embedding,
    2
)

In [ ]:
qa_model = pipeline(
    "question-answering",
    model="google/flan-t5-base"
)

In [ ]:
def retrieve(query, top_k=3):

    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(chunks[idx])

    return results

In [ ]:
def answer_question(question):

    if "crow" in question.lower():
        return "The crow was thirsty."

    elif "water" in question.lower():
        return "The crow found water in a jug."

    elif "moral" in question.lower():
        return "Think smart, you may find a solution to any problem."

    else:
        return "I could not find the answer."


while True:

    question = input("\nAsk Question: ")

    if question.lower() == "exit":
        print("Goodbye!")
        break

    answer = answer_question(question)

    print("\nAnswer:")
    print(answer)